# Stage → Raw volume landing

Run this notebook once, before the `03_raw_to_bronze` → `04_bronze_to_silver` → `05_silver_to_gold` Run All chain. It is the platform boundary: local ingestion writes `./landing`, the user uploads that tree through the Catalog UI, and this notebook copies staged envelopes into the Spark-visible UC volume.

## Contract

Staged files retain their relative `measurements/...` (or metadata) path. Existing destination files are skipped, every landed file is verified with `dbutils.fs.ls`, its in-file `batch_id` is printed, and landed staging paths are printed for local cleanup. Workspace files are read-only from serverless compute, so this notebook never deletes them.

In [ ]:
import json
import os

STAGING_ROOT_API = "/Users/dammifabz@gmail.com/openaq_staging"
STAGING_ROOT = "/Workspace" + STAGING_ROOT_API
VOLUME_ROOT = "/Volumes/workspace/raw/openaq_landing"

def recursive_files(path):
    result = []
    for root, _, names in os.walk(path):
        for name in names:
            if name.endswith(".json"):
                result.append(os.path.join(root, name))
    return sorted(result)

def exists(path):
    parent, name = path.rsplit("/", 1)
    return any(item.name.rstrip("/") == name for item in dbutils.fs.ls(parent + "/"))

staged_files = recursive_files(STAGING_ROOT)
print(f"staged_files={len(staged_files)}")
for staged_path in staged_files:
    relative_path = staged_path[len(STAGING_ROOT):].lstrip("/")
    try:
        with open(staged_path, "r", encoding="utf-8") as source:
            content = source.read()
    except Exception as exc:
        print(f"OPEN_FAILED path={staged_path} error={exc!r}")
        raise
    envelope = json.loads(content)
    destination = f"{VOLUME_ROOT}/{relative_path}"
    if exists(destination):
        envelope = json.loads(dbutils.fs.head(destination, 10 * 1024 * 1024))
        print(f"SKIP existing destination: {destination} batch_id={envelope['ingestion']['batch_id']}")
    else:
        dbutils.fs.put(destination, content, overwrite=False)
        if not exists(destination):
            raise RuntimeError(f"destination not visible after copy: {destination}")
        envelope = json.loads(dbutils.fs.head(destination, 10 * 1024 * 1024))
        print(f"LANDED: {destination} batch_id={envelope['ingestion']['batch_id']}")
print("stage-and-land complete; landed staging paths:")
for staged_path in staged_files:
    print(staged_path)